<a href="https://colab.research.google.com/github/Paulopurcino22/Case_Startup/blob/main/Ragexcel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
!pip install llama-index
!pip install llama-parse

In [74]:
import nest_asyncio

from llama_index.llms.openai import OpenAI
from llama_index.core import VectorStoreIndex
from IPython.display import Image, Markdown

from llama_parse import LlamaParse

from llama_index.core.node_parser import MarkdownElementNodeParser

In [75]:
nest_asyncio.apply()

In [76]:
import os

from google.colab import userdata


os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

llm_o1 = OpenAI(model="o1-mini")
llm_gpt4o_mini = OpenAI(model="gpt-4o-mini")
llm_o1_preview = OpenAI(model="o1-preview")

In [77]:
from google.colab import userdata


parser = LlamaParse(
    api_key=userdata.get('LLAMA_CLOUD_API_KEY'),
    result_type="markdown",
)

documents = parser.load_data("/content/base_rh.xlsx")

Started parsing the file under job_id 59937467-0e67-40b0-aad9-67b60e5bb78d


In [78]:
len(documents)

1

In [79]:
node_parser = MarkdownElementNodeParser(llm=llm_gpt4o_mini, num_workers=4)

In [80]:
nodes = node_parser.get_nodes_from_documents(documents[:10])

1it [00:00, 163.24it/s]


In [81]:
base_nodes, objects = node_parser.get_nodes_and_objects(nodes)

In [82]:
len(nodes), len(base_nodes), len(objects)

(3, 1, 1)

In [83]:
print(objects[0].get_content())

This table contains employee data, including their age, travel frequency, distance to work, education level, job satisfaction (E-Sat), gender, marital status, salary, number of companies worked for, overtime status, percentage of salary increase, number of company shares, career duration, training hours, work-life balance, tenure at the company, years in the same position, years since last promotion, and years with the same manager. It also indicates whether the employee has left the company.,
with the following columns:
- ID: None
- Funcionário_deixou_a_empresa: None
- Idade: None
- Frequência de Viagens: None
- Distância_do_trabalho: None
- Formação: None
- E-Sat: None
- Gênero: None
- Estado_Civil: None
- Salário: None
- Qte_Empresas_Trabalhadas: None
- Faz_hora_extras?: None
- Perc_de_aumento: None
- Qte_ações_da_empresa: None
- Tempo_de_carreira: None
- Horas_de_treinamento: None
- Equilibrio_de_Vida: None
- Tempo_de_empresa: None
- Anos_no_mesmo_cargo: None
- Anos_desde_a_ultima_

In [84]:
# dump both indexed tables and page text into the vector index
recursive_index = VectorStoreIndex(nodes=base_nodes + objects, llm=llm_gpt4o_mini)

recursive_query_engine_o1 = recursive_index.as_query_engine(
    similarity_top_k=5, llm=llm_o1
)

recursive_query_engine_o1_preview = recursive_index.as_query_engine(
    similarity_top_k=5, llm=llm_o1_preview
)

recursive_query_engine_gpt4o_mini = recursive_index.as_query_engine(
    similarity_top_k=5, llm=llm_gpt4o_mini
)

In [88]:
query = "Quantos funcionários com 8 anos de carreira deixaram a empresa?"

response_recursive_gpt4o_mini = recursive_query_engine_gpt4o_mini.query(query)

In [89]:
print("----------------------RESPONSE WITH GPT4O-MINI----------------------")
display(Markdown(f"{response_recursive_gpt4o_mini}"))

----------------------RESPONSE WITH GPT4O-MINI----------------------


Para determinar quantos funcionários com 8 anos de carreira deixaram a empresa, é necessário contar as entradas na tabela onde o tempo de carreira é igual a 8 anos e a coluna que indica se o funcionário deixou a empresa está marcada como "Sim". 

Após a análise, o número de funcionários que atendem a esses critérios é 20.